# Tutorial: Building an AC Circuit Analyzer

In [73]:
from __future__ import annotations

import abc
from dataclasses import dataclass, field
from math import atan2, cos, degrees, pi, radians, sin
from typing import Self

from sympy import Symbol, nsolve

This chapter is a practical tutorial that will build your AC circuit analysis and programming skills. The goal is to develop a reusable software tool that can solve for all voltages and currents of any steady-state AC circuit consisting of passive components: sinusoidal voltage and current sources, resistors, inductors, and capacitors. To keep the program simple, there is no need to build a user interface. The input circuit to analyze can be built programmatically, by defining components and how they're connected.

Let's first discuss how we can model the problem and design our solution. Jumping right into a solution would not be as interesting, useful, or realistic; the solutions to real-world problems are not handed out and often require thinking through the problem from the beginning before writing any code.

## Design Discussion

Let's start with the building blocks that will be used to construct and represent the circuit. It makes sense to use classes (or structs, in programming languages like Rust and Go) to represent our two-terminal electric components, but we also need to think about how to represent the connections between those components. Consider:

- Does the orientation with which components are connected (positive versus negative terminals) matter?

    Yes; it matters in order to establish a convention for voltage polarity and current direction, for the sake of "measuring" the solved voltage/current of a component, as well as for specifying the voltage/current of a sinusoidal voltage/current source.

- Is it best to model the components as being connected by wires, or by nodes?

    Wires are the more intuitive physical representation, but if wires were to be used, at least $n - 1$ of them would be needed to connect $n$ component terminals to each other. The situation can be simplified by using a "wire" with $n$ ends to connect to $n$ terminals. This "$n$-ended wire" is simply a node. Using nodes is also the clear choice to be able to apply KCL easily.

For convenience when applying KCL, each node can list all components attached to it. However, that would not allow us to represent which of the component's terminals&mdash;positive or negative&mdash;are attached to a given node. To address this, each component can have attributes `positive` and `negative`, specifying which terminal each node is attached to.

We know that any AC circuit can be analyzed by applying KCL and solving a linear system of complex-valued phasor equations in terms of the node voltages. Formulating that system of equations is routine when analyzing circuits by hand, but formalizing that process to be done by a computer requires more thought. Consider:

- What does "summing the currents flowing out of each node" mean when developing a program? Does that mean finding a way to represent equations programmatically and then convert them to the standard format for linear systems, $A \mathbf{x} = \mathbf{b}$, with which solvers are compatible? Or does that mean populating the matrix $A$ and vector $\mathbf{b}$ directly?

    For convenience, we can indeed use a symbolic math library (such as Python's [`sympy`](https://www.sympy.org/en/index.html) and its [`nsolve`](https://docs.sympy.org/latest/modules/solvers/solvers.html#sympy.solvers.solvers.nsolve) solver) to "write" and solve the equations, without needing to manually put them into the $A \mathbf{x} = \mathbf{b}$ format; `sympy` can do the heavy lifting.

- How does one determine when the supernode technique must be used, and apply it programmatically?

    The system of equations can be generalized&mdash;boiling down the circuit analysis problem to its simplest laws&mdash;such that the supernode technique need not be explicitly applied.

    In regular KCL, we sum the currents out of each node *in terms of the node voltages*. When this cannot be done, such as for nodes $a$ and $b$ connected by voltage source $V_{s2}$ in {ref}`fig_3_1`, we must combine them and sum the currents out of the resulting supernode $ab$. The voltage source's characteristic equation, $V_{s2} = V_a - V_b$, makes up for the missing KCL equation.

    Instead, let's sum the currents out of each node, and&mdash;rather than putting those currents in terms of node voltages&mdash;add the characteristic equation for each connected component to the system of equations. This eliminates the need for the supernode technique. This approach is atypical when solving equations by hand, as it requires writing out a larger initial system of equations, but this is a non-issue for a computer.

For every node and component, there will be one unknown (the node's voltage or the component's current) and one equation (KCL or the component's characteristic equation). However, there are two exceptions to this. Firstly, in an $n$-node circuit, only $n - 1$ KCL equations are required; the $n$th adds no new information to the system of equations. And secondly, because voltages are relative, a voltage reference is needed to prevent the system from having an infinite number of solutions. Both these points can be addressed by making the $n$th node a ground node, which contributes $V = 0$ to the system of equations rather than KCL.

Now that we have formalized the problem and considered the design options, we're ready to implement. Give it a shot, consulting the following sections for ideas if desired (they use Python, but feel free to use any programming language).

## Data Model

To start developing our program, we need a way to represent voltage and current values&mdash;for the sake of representing solved values, as well as for specifying the voltage/current of a sinusoidal voltage/cur&shy;rent source. Python has built-in support for complex numbers, with the `complex` type and `1j` literal, but it is more common outside of numerical computation to work with phasors in polar form. So, let's create a `Phasor` class (polar form), with methods for converting to and from complex numbers:

In [ ]:
@dataclass
class Phasor:
    magnitude: float
    phase_deg: float = 0.0

    @property
    def phase_rad(self) -> float:
        return radians(self.phase_deg)

    @classmethod
    def from_complex(cls, value: complex) -> Self:
        return cls(
            magnitude=abs(value),
            phase_deg=degrees(atan2(value.real, value.imag)),
        )

    def to_complex(self) -> complex:
        return self.magnitude * (
            cos(self.phase_deg) + 1j * sin(self.phase_deg)
        )

    def __repr__(self) -> str:
        return f"{self.magnitude:.6f} @ {self.phase_deg:.3f}°"

We also need a way to represent the duality of voltages and currents, either as unknown variables (`Symbol`s) in our `sympy` equations to solve for, or as solved output values. To do so, let's create `Voltage` and `Current` classes, each with both a `variable` of type `Symbol` and a `value` of type `Phasor`. The `Symbol` is automatically named depending on whether it represents a node's voltage or a component's current, allowing both us and `sympy` to meaningfully distinguish them. These traits are common to both `Voltage` and `Current`, so we create a shared base class for them (`_Symbolic`). Note that a positive value of current indicates flow from the positive terminal to negative.

In [ ]:
@dataclass(kw_only=True)
class _Symbolic:
    value: Phasor | None = None

    @property
    def variable(self) -> Symbol:
        return Symbol(str(self))

@dataclass
class Voltage(_Symbolic):
    node_name: str

    def __str__(self) -> str:
        return f"(node {self.node_name} voltage)"

@dataclass
class Current(_Symbolic):
    component_name: str

    def __str__(self) -> str:
        return f"(component {self.component_name} current)"

## Building Blocks

Now let's create the building blocks that we'll use to represent the circuit: nodes and components. First we create the `Node` class. Every node contributes a KCL equation and has a `Voltage` that is free to vary, except the ground node (`is_ground=True`), whose voltage is fixed at zero.

In [76]:
@dataclass(repr=False)
class Node:
    name: str
    is_ground: bool = False
    voltage: Voltage = field(init=False)
    connected_components: list[_BaseComponent] = field(
        init=False
    )

    def __post_init__(self):
        self.voltage = Voltage(node_name=self.name)
        # Appended to by `BaseComponent.__post_init__`s:
        self.connected_components = []

    def __hash__(self) -> int:
        return hash(self.name)

    @property
    def equation(self):
        if self.is_ground:
            return self.voltage.variable
        else:
            return sum(
                c.current.variable
                * (1 if c.positive is self else -1)
                for c in self.connected_components
            )

Unlike nodes, which are universal, we have multiple types of component. However, they have much in common: they all have current flowing from a positive terminal to a negative terminal, and they generally have a characteristic equation describing the relationship between that electric current and the voltage difference across the component (between the two connected nodes). To avoid having to write the same code over and over, we create a `_BaseComponent`, of which every type of component will be a subclass. Each component must implement the `equation` method (which really returns a `sympy` *expression*, used as the LHS in an equation whose RHS is zero). The frequency (`frequency_rad_per_s`) is passed to each `equation` method because it is a global property of the circuit, but will be needed in the next section by the `Inductor` and `Capacitor` components to calculate reactance and admittance.

In [77]:
@dataclass(repr=False)
class _BaseComponent(abc.ABC):
    name: str
    negative: Node
    positive: Node
    current: Current = field(init=False)

    def __post_init__(self):
        self.current = Current(component_name=self.name)
        for node in self._connected_nodes:
            node.connected_components.append(self)

    @property
    def _connected_nodes(self) -> list[Node]:
        return [self.negative, self.positive]

    @abc.abstractmethod
    def equation(self, frequency_rad_per_s: float):
        raise NotImplementedError

    @property
    def voltage_difference(self):
        return (
            self.positive.voltage.variable
            - self.negative.voltage.variable
        )

## Component Library

Now that we have created our framework, we are able to implement our components and focus on the characteristic equation of each:

In [ ]:
@dataclass(repr=False)
class VoltageSource(_BaseComponent):
    value: Phasor

    def equation(self, frequency_rad_per_s: float):
        return (
            self.voltage_difference - self.value.to_complex()
        )

@dataclass(repr=False)
class CurrentSource(_BaseComponent):
    value: Phasor

    def equation(self, frequency_rad_per_s: float):
        return self.current.variable - self.value.to_complex()

@dataclass(repr=False)
class Resistor(_BaseComponent):
    resistance_ohm: float

    def equation(self, frequency_rad_per_s: float):
        return (
            self.voltage_difference
            - self.current.variable * self.resistance_ohm
        )

@dataclass(repr=False)
class Inductor(_BaseComponent):
    inductance_h: float

    def equation(self, frequency_rad_per_s: float):
        reactance = (
            1j * frequency_rad_per_s * self.inductance_h
        )
        return (
            self.voltage_difference
            - self.current.variable * reactance
        )

@dataclass(repr=False)
class Capacitor(_BaseComponent):
    capacitance_f: float

    def equation(self, frequency_rad_per_s: float):
        admittance = 1j * (
            frequency_rad_per_s * self.capacitance_f
        )
        return (
            self.voltage_difference * admittance
            - self.current.variable
        )

Rather than using reactance for the capacitor, we use its inverse&mdash;admittance&mdash;as it conveniently allows the circuit's frequency to be set to zero to model the circuit under DC conditions, without encountering a division-by-zero error.

## The Circuit Solver

All that's left to do is create a circuit solver to determine the voltage and current values. This is done by:

1. Passing all the circuit's components to the solver.
2. Extracting all the circuit's nodes (taken from the components' `positive` and `negative` terminals).
3. Collecting the unknowns (voltages and currents) and equations (KCL and characteristic equations) from the nodes and components.
4. Solving the equations for the unknowns using `sympy`'s `nsolve`.
5. Assigning the solved values to their respective nodes and components.

In [79]:
@dataclass(repr=False)
class AcCircuitSolver:
    frequency_hz: float
    components: list[_BaseComponent]

    def solve(self) -> None:
        connected_nodes = self._get_connected_nodes()
        symbolics = self._get_symbolics(connected_nodes)
        equations = self._get_equations(connected_nodes)
        unknowns = [s.variable for s in symbolics]
        initial_guess = [0.0 for s in symbolics]
        solution = nsolve(equations, unknowns, initial_guess)
        for unknown, solved_value in zip(symbolics, solution):
            unknown.value = Phasor.from_complex(
                complex(solved_value)
            )

    def _get_symbolics(
        self, connected_nodes: list[Node]
    ) -> list[_Symbolic]:
        symbolics: list[_Symbolic] = []
        for n in connected_nodes:
            symbolics.append(n.voltage)
        for c in self.components:
            symbolics.append(c.current)
        return symbolics

    def _get_equations(self, connected_nodes: list[Node]):
        equations = []
        for n in connected_nodes:
            equations.append(n.equation)
        frequency_rad_per_s = 2 * pi * self.frequency_hz
        for c in self.components:
            equations.append(c.equation(frequency_rad_per_s))
        return equations

    def _get_connected_nodes(self) -> list[Node]:
        nodes: list[Node] = []
        for component in self.components:
            nodes += component._connected_nodes
        # Deduplicate (requires `Node.__hash__`):
        nodes = list(set(nodes))
        return nodes

## Example Usage

We can apply our solver to the example circuit of the last chapter's {ref}`fig_3_1`. First, we specify the circuit's nodes&mdash;the positive and negative terminals of both AC voltage sources&mdash;giving each a unique name:

In [80]:
vs1p = Node("Vs1p")
vs1n = Node("Vs1n", is_ground=True)
a = Node("Vs2p")
b = Node("Vs2n")

Now we specify the two AC voltage sources, the two resistors, the inductor, and the capacitor:

In [81]:
vs1 = VoltageSource("Vs1", vs1n, vs1p, Phasor(magnitude=1))
vs2 = VoltageSource("Vs2", b, a, Phasor(magnitude=2))
r1 = Resistor("R1", b, vs1p, resistance_ohm=4)
r2 = Resistor("R2", vs1n, a, resistance_ohm=5)
l = Inductor("L", a, vs1p, inductance_h=6)
c = Capacitor("C", vs1n, b, capacitance_f=7)

Next, we pass the components to the solver, specify the circuit's frequency (or $0 \ \mathrm{Hz}$ for DC conditions), and run the solver:

In [82]:
solver = AcCircuitSolver(
    frequency_hz=(3 / (2 * pi)),
    components=[vs1, vs2, r1, r2, l, c],
)
solver.solve()

We can access the solved voltage and current values through the original node and component objects:

In [83]:
print(f"{a.voltage.value = }")
print(f"{b.voltage.value = }")
print(f"{vs2.current.value = }")

a.voltage.value = 2.003 @ 89.794°
b.voltage.value = 0.008 @ 19.092°
vs2.current.value = 0.405 @ -82.293°


This chapter is available [as a runnable Python notebook](https://github.com/keeganmjgreen/how_phasors_work/blob/main/4_building_an_ac_circuit_analyzer.ipynb) and [as a Python script](https://github.com/keeganmjgreen/how_phasors_work/blob/main/4_building_an_ac_circuit_analyzer/ac_circuit_solver.py).

## Additional Exercises

1. Add a plotting feature to the circuit analyzer (e.g., using Python's `matplotlib`).

2. Add the ability to import and analyze the circuit from a file or from a circuit diagram.

3. Consider how you would adapt the circuit analyzer to support transient simulations and active components (diodes, transistors, etc.).

See [here](https://github.com/keeganmjgreen/circuit_simulator) for some inspiration.